# Test SFT sur dataset medical

### DATASET

- Load dataset
- Preprocessing pour avoir le format (prompt/completion) nécessaire pour SFTTrainer

In [1]:
from pathlib import Path
from datasets import load_dataset, load_from_disk

# utility code - télécharge le dataset si pas déjà présent sur le disque
# et le charge localement

HF_DATASET_ID = "FreedomIntelligence/medical-o1-reasoning-SFT"
HF_CONFIG = "en"
DATASET_DIR = Path("../datasets").resolve()

# Ensure target directory exists
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# "Empty" means no files/folders inside
is_empty = not any(DATASET_DIR.iterdir())

if is_empty:
    print(f"{DATASET_DIR} is empty -> downloading from Hub and saving to disk...")
    dataset = load_dataset(HF_DATASET_ID, HF_CONFIG)
    dataset.save_to_disk(str(DATASET_DIR))
else:
    print(f"{DATASET_DIR} is not empty -> loading from disk...")
    dataset = load_from_disk(str(DATASET_DIR))

print(dataset)

/home/benjamin.deporte/GenAI_Projects/datasets is not empty -> loading from disk...
DatasetDict({
    train: Dataset({
        features: ['Question', 'Complex_CoT', 'Response'],
        num_rows: 19704
    })
})


In [2]:
from pprint import pprint

def preprocess_function(example):
    return {
        "prompt": [{"role": "user", "content": example["Question"]}],
        "completion": [
            {"role": "assistant", "content": f"<think>{example['Complex_CoT']}</think>{example['Response']}"}
        ],
    }

dataset = dataset.map(preprocess_function, remove_columns=["Question", "Response", "Complex_CoT"])
pprint(next(iter(dataset["train"])))

{'completion': [{'content': "<think>Okay, let's see what's going on here. "
                            "We've got sudden weakness in the person's left "
                            'arm and leg - and that screams something '
                            'neuro-related, maybe a stroke?\n'
                            '\n'
                            "But wait, there's more. The right lower leg is "
                            'swollen and tender, which is like waving a big '
                            'flag for deep vein thrombosis, especially after a '
                            'long flight or sitting around a lot.\n'
                            '\n'
                            "So, now I'm thinking, how could a clot in the leg "
                            'end up causing issues like weakness or stroke '
                            'symptoms?\n'
                            '\n'
                            "Oh, right! There's this thing called a "
                            "paradox

In [3]:
# subset first, then split
N_MAX = 100
SPLIT_SEED = 42
VAL_RATIO = 0.1

base_train = dataset["train"]

if N_MAX <= 0:
    raise ValueError("N_MAX must be > 0")

n_subset = min(N_MAX, len(base_train))
subset_train = base_train.shuffle(seed=SPLIT_SEED).select(range(n_subset))

split = subset_train.train_test_split(
    test_size=VAL_RATIO,
    seed=SPLIT_SEED,
    shuffle=True,
)

train_ds = split["train"]
val_ds = split["test"]

print(f"Original train size: {len(base_train)}")
print(f"Subset size:         {len(subset_train)}")
print(f"Train size:          {len(train_ds)}")
print(f"Validation size:     {len(val_ds)}")

Original train size: 19704
Subset size:         100
Train size:          90
Validation size:     10


### MODELE

In [4]:
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model
import torch

In [5]:
# utility function pour calculer le nombre de paramètres trainables
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

In [6]:
# utility code - idem, télécharge le modèle si pas déjà présent sur le disque
# et le sauve localement

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
MODEL_DIR = Path("../models/qwen2_5_3b_instruct").resolve()

MODEL_DIR.mkdir(parents=True, exist_ok=True)
is_empty = not any(MODEL_DIR.iterdir())

if is_empty:
    print(f"{MODEL_DIR} is empty -> downloading model/tokenizer from Hub and saving to disk...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,   # ou torch.float16 selon ton GPU
        device_map="auto"
    )
    tokenizer.save_pretrained(str(MODEL_DIR))
    model.save_pretrained(str(MODEL_DIR))
else:
    print(f"{MODEL_DIR} is not empty -> loading model/tokenizer from disk...")
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

/home/benjamin.deporte/GenAI_Projects/models/qwen2_5_3b_instruct is not empty -> loading model/tokenizer from disk...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [7]:
# base_model=AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
base_model = model

# Before LoRA (full model)
full_trainable, full_total = count_params(base_model)

print(f"Full FT trainable: {full_trainable:,}")

Full FT trainable: 3,085,938,688


In [8]:
# using PEFT and LoRA pour réduire le besoin en VRAM

from peft import LoraConfig
RANG=4

peft_config=LoraConfig(
    task_type="CAUSAL_LM",   # decrit quel type de modèle on adapte
    inference_mode=False, # ordonne que les paramètres soient entrainables
    r=RANG, # rang !
    lora_alpha=2*RANG, # scaling factor pour l'update LoRA
    lora_dropout=0.05, # dropout pour le LoRA
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # LoRA sur les têtes d'attention
)

In [9]:
lora_model=get_peft_model(base_model, peft_config)

# After LoRA wrapping
lora_trainable, lora_total = count_params(lora_model)

reduction_vs_full_ft = 100 * (1 - (lora_trainable / full_trainable))
trainable_pct_of_total = 100 * (lora_trainable / lora_total)

print(f"Full FT trainable: {full_trainable:,}")
print(f"LoRA trainable:    {lora_trainable:,}")
print(f"Reduction:         {reduction_vs_full_ft:.2f}%")
print(f"LoRA trainable %:  {trainable_pct_of_total:.4f}%")

Full FT trainable: 3,085,938,688
LoRA trainable:    1,843,200
Reduction:         99.94%
LoRA trainable %:  0.0597%


### EVALUATION metrics

In [10]:
import numpy as np

# Optional metric: token-level accuracy (ignores -100 labels)
def preprocess_logits_for_metrics(logits, labels):
    # logits est typiquement (batch, seq_len, vocab_size)
    if isinstance(logits, tuple):
        logits = logits[0]
    # renvoie (batch, seq_len) avec le predicted token id
    return logits.argmax(dim=-1)

def compute_metrics(eval_pred):
    preds, labels = eval_pred  # numpy arrays
    # next-token shift
    preds = preds[:, :-1].reshape(-1)
    labels = labels[:, 1:].reshape(-1)

    mask = labels != -100
    if mask.sum() == 0:
        return {"token_accuracy": 0.0}

    # compte 1 pour les bons tokens prédits, et la moyenne
    acc = (preds[mask] == labels[mask]).astype(np.float32).mean().item()
    return {"token_accuracy": acc}

In [11]:
# 3) Enable eval during training
sft_args = SFTConfig(
    output_dir="outputs/sft-medical",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,      # important pour ma pauvre 3080 Ti 12 Go
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=3,
    
    # log each epoch
    logging_strategy="epoch",
    logging_dir="./logs",
    # logging_steps=10,
    save_steps=100,
    
    # Optional: use tensorboard for better visualization
    # report_to=["tensorboard"],

    # max_seq_length=512,
    gradient_checkpointing=True,
    bf16=True,                         # or fp16=True on your hardware

    eval_strategy="steps",             # if your version expects evaluation_strategy, use that key instead
    eval_steps=100,
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    eval_accumulation_steps=8,         # helps eval memory
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
trainer = SFTTrainer(
    model=lora_model,  # use the already reduced model
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    # processing_class=tokenizer
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

In [13]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Token Accuracy
18,1.734499,1.658380,0.586058


TrainOutput(global_step=18, training_loss=1.7727834913465712, metrics={'train_runtime': 110.1212, 'train_samples_per_second': 2.452, 'train_steps_per_second': 0.163, 'total_flos': 2745802064977920.0, 'train_loss': 1.7727834913465712})

### QUELQUES TESTS

In [14]:
def parse_completion(completion_text):
    """
    Parse completion text to extract reasoning and final answer.
    Format: <think>REASONING</think>FINAL_ANSWER
    
    Returns: (reasoning_text, final_answer_text)
    """
    if "<think>" in completion_text and "</think>" in completion_text:
        start_idx = completion_text.find("<think>") + len("<think>")
        end_idx = completion_text.find("</think>")
        reasoning = completion_text[start_idx:end_idx].strip()
        final_answer = completion_text[end_idx + len("</think>"):].strip()
    else:
        # No think tags, treat entire text as final answer
        reasoning = ""
        final_answer = completion_text.strip()
    
    return reasoning, final_answer

In [15]:
import pandas as pd
import torch
from transformers import AutoTokenizer

N_SAMPLES = 5
SAMPLE_SEED = 123
MAX_NEW_TOKENS = 256

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_for_eval = trainer.model
model_for_eval.eval()

rng = np.random.default_rng(SAMPLE_SEED)
n_eval = min(N_SAMPLES, len(val_ds))
sample_indices = rng.choice(len(val_ds), size=n_eval, replace=False).tolist()

print(f"Sampling {n_eval} examples from validation set of size {len(val_ds)}")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Sampling 5 examples from validation set of size 10


In [16]:
results = []

for idx in sample_indices:
    ex = val_ds[int(idx)]

    # prompt already stored as chat messages
    messages = ex["prompt"]
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model_for_eval.device)

    with torch.no_grad():
        generated = model_for_eval.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    gen_tokens = generated[0][prompt_len:]
    pred_completion = tokenizer.decode(gen_tokens, skip_special_tokens=True)

    # Parse ground truth
    gt_full = ex["completion"][0]["content"]
    gt_reasoning, gt_final_answer = parse_completion(gt_full)

    # Parse prediction
    pred_reasoning, pred_final_answer = parse_completion(pred_completion)

    results.append(
        {
            "idx": int(idx),
            "prompt": messages[0]["content"],
            "gt_reasoning": gt_reasoning,
            "gt_final_answer": gt_final_answer,
            "pred_reasoning": pred_reasoning,
            "pred_final_answer": pred_final_answer,
            "gt_full": gt_full,
            "pred_full": pred_completion,
        }
    )

print(f"Built {len(results)} comparisons with parsed components.")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Built 5 comparisons with parsed components.


In [17]:
preview_df = pd.DataFrame(results)[
    ["idx", "prompt", "gt_reasoning", "gt_final_answer", "pred_reasoning", "pred_final_answer"]
].copy()

# Truncate for display
preview_df["prompt"] = preview_df["prompt"].str.slice(0, 80) + "..."
preview_df["gt_reasoning"] = preview_df["gt_reasoning"].str.slice(0, 100) + "..."
preview_df["gt_final_answer"] = preview_df["gt_final_answer"].str.slice(0, 100) + "..."
preview_df["pred_reasoning"] = preview_df["pred_reasoning"].str.slice(0, 100) + "..."
preview_df["pred_final_answer"] = preview_df["pred_final_answer"].str.slice(0, 100) + "..."

preview_df

,idx,prompt,gt_reasoning,gt_final_answer,pred_reasoning,pred_final_answer
0,8,After resuscitating a patient who has bleeding...,"So, when a patient has bleeding esophageal var...",After resuscitating a patient with bleeding es...,...,After successfully resuscitating a patient wit...
1,7,"A 37-year-old man presents with dull, continuo...","So, we've got a 37-year-old guy complaining of...",The clinical presentation and investigative fi...,...,Based on the clinical presentation and imaging...
2,0,Two days after vaginal delivery of a healthy n...,"Alright, so we've got a 32-year-old woman who ...","In this case, given the history of significant...",...,"Given the clinical scenario described, it appe..."
3,9,In a 12-year-old boy with sickle cell disease ...,"Okay, so we've got a 12-year-old boy with sick...",In a 12-year-old boy with sickle cell disease ...,...,In a 12-year-old boy with sickle cell disease ...
4,4,A 29-year-old woman presents with painful swel...,"Okay, let's think about this. A 29-year-old wo...","Based on the clinical features presented, the ...",...,Based on the clinical features you've describe...


In [18]:
for i, r in enumerate(results, start=1):
    print("=" * 120)
    print(f"Sample {i} | Validation index: {r['idx']}")
    print("\n[PROMPT]")
    print(r["prompt"])
    print("\n[GROUND TRUTH - Reasoning]")
    print(r["gt_reasoning"] if r["gt_reasoning"] else "(no reasoning tags)")
    print("\n[GROUND TRUTH - Final Answer]")
    print(r["gt_final_answer"])
    print("\n[PREDICTED - Reasoning]")
    print(r["pred_reasoning"] if r["pred_reasoning"] else "(no reasoning tags)")
    print("\n[PREDICTED - Final Answer]")
    print(r["pred_final_answer"])
    print()

Sample 1 | Validation index: 8

[PROMPT]
After resuscitating a patient who has bleeding oesophageal varices, what is the first treatment that should be administered?

[GROUND TRUTH - Reasoning]
So, when a patient has bleeding esophageal varices, it's kind of scary because it can be really life-threatening. Most of the time, this is due to portal hypertension, which I've heard is often connected to cirrhosis. That's why we can't waste any time once the patient is resuscitated - controlling the bleeding is super urgent to avoid them going into shock or something.

Alright, so what can be done to control the bleeding? There are a few options, but initially, it's critical to stabilize things fast. We definitely have to think about reducing that portal pressure. Hmm, I remember that there are some medications, like octreotide or terlipressin, that are used for this purpose. They're vasoactive drugs and they're pretty quick at helping with hemostasis, which sounds like exactly what's needed 